# 环节 06 · DPO 家族与离线偏好优化（配套 Notebook）

> 配套长文：[环节06-DPO家族与离线偏好优化详解.md](./环节06-DPO家族与离线偏好优化详解.md)
> 定位：从 RLHF 闭式解出发，验证 `Z(x)` 被消掉、跑 DPO 损失与梯度权重、对比家族变体。全部**纯 Python 标准库**。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 Z(x) 消掉 | §2 / §8 | 不同 Z(x) 得到同一个 Δ |
| §2 DPO 损失与权重 | §3 / §8 | `σ(−Δ)` 自动降低已分对样本的权重 |
| §3 β 语义 | §3 | β 小 → 永不饱和、猛推；β 大 → 很快饱和 |
| §4 家族变体 | §5 | DPO / IPO / SimPO 的损失形态差异 |
| §5 长度偏置 | §6 | 长度归一化到底修了什么 |


## 1. 把闭式解代回 BT：`Z(x)` 为什么会消失

RLHF 闭式最优解 `π*(y|x) = π_ref(y|x)·exp(r(x,y)/β) / Z(x)`，反解奖励：

```
r(x, y) = β·log( π*(y|x) / π_ref(y|x) ) + β·log Z(x)
```

代进 BT 的 `P = σ(r_w − r_l)`：`β·log Z(x)` 是**只依赖 x 的常数**，配对相减时**自动消掉**。


In [ ]:
import math

beta, lw, ll = 0.3, 1.2, 0.4        # log(π_θ/π_ref) 在 chosen / rejected 上的值
print("取任意不同的 Z(x)，看奖励差 Δ = r_w − r_l 变不变：")
for Z in (1.0, 7.5, 1e6):
    r_w = beta * lw + beta * math.log(Z)
    r_l = beta * ll + beta * math.log(Z)
    print(f"  Z(x) = {Z:>9}   r_w − r_l = {r_w - r_l:.6f}")
print("→ 三者完全相同：配分函数在训练目标里根本不需要出现。")


## 2. DPO 损失：把"训 RM"和"跑 RL"合并成一步监督学习

```
L_DPO = − E[ log σ( Δ ) ]，  Δ = β·[ log(π_θ(y_w)/π_ref(y_w)) − log(π_θ(y_l)/π_ref(y_l)) ]
```

- **只要 2 个模型**（策略 + 参考），没有 RM、没有 rollout、没有 critic；
- 本质是**隐式奖励**下的监督学习：`r_θ = β·log(π_θ/π_ref)`。

梯度：`∇L = −σ(−Δ)·∇Δ` —— **权重就是 `σ(−Δ)`**，已分对的样本自动降权。


In [ ]:
sig = lambda x: 1 / (1 + math.exp(-x))
print(f"{'β':>5} {'g':>5} {'Δ = β·g':>9} {'loss':>9} {'权重 σ(−Δ)':>12} {'隐式奖励差':>12}")
for beta in (0.01, 0.1, 0.5):
    for g in (-1.0, 0.0, 1.0, 3.0):
        d = beta * g
        print(f"{beta:>5} {g:>5.1f} {d:>9.3f} {-math.log(sig(d)):>9.4f}"
              f" {sig(-d):>12.3f} {beta * g:>12.3f}")
print("→ β 小 → Δ 被压小 → 永不饱和、一直猛推；β 大 → 迅速饱和、更新变慢。")


## 3. 梯度权重曲线：`σ(−Δ)` 就是"还差多少要学"

Δ 很大（模型已经把 chosen 分得远高于 rejected）→ 权重 → 0；
Δ 很负（学反了）→ 权重 → 1，全力纠正。


In [ ]:
print("DPO 梯度权重 σ(−Δ)：")
for d in (-3.0, -1.0, 0.0, 1.0, 3.0, 6.0):
    bar = "#" * round(sig(-d) * 40)
    print(f"  Δ = {d:>+5.1f}  权重 = {sig(-d):.4f}  {bar}")
print("→ 权重永远 > 0：DPO 不会像 hinge 那样「彻底放弃」某条样本。")


## 4. 家族变体：都在改"目标差距"与"要不要参考模型"

| 方法 | 参考模型 | 关键改动 |
|---|---|---|
| **DPO** | 需要 | 基线 |
| **IPO** | 需要 | 把 log-sigmoid 换成平方损失，目标差距锚在 `1/(2β)`，抗过拟合 |
| **SimPO** | **不需要** | `β/|y|·logπ` 长度归一化 + target margin γ |
| **ORPO** | **不需要** | SFT 损失 + odds-ratio 偏好项，一步到位（无参考模型） |
| **KTO** | **不需要** | 不要求成对，好/坏样本各自二分类（可用点赞点踩） |


In [ ]:
beta, gamma = 0.1, 0.5
print(f"{'Δ':>6} {'DPO: −logσ(βΔ)':>17} {'IPO: (βΔ−0.5)²':>18} {'SimPO: −logσ(βΔ−γ)':>20}")
for D in (-2.0, -1.0, 0.0, 1.0, 2.0):
    dpo = -math.log(sig(beta * D))
    ipo = (beta * D - 0.5) ** 2
    simpo = -math.log(sig(beta * D - gamma))
    print(f"{D:>6.1f} {dpo:>17.4f} {ipo:>18.4f} {simpo:>20.4f}")
print("→ 差别只在「目标差距设在哪」+「要不要参考模型」；数据工程比选变体更重要。")


## 5. 长度偏置：DPO 最经典的坑

偏好对里若 chosen 总是更长，DPO 会学出「越长越好」。**根因是 log 概率是负数、回答越长累加越负**：

```
log π(y) = Σ_t log π(y_t)   → 长度越长，这个数越小
```


In [ ]:
# 同一条"质量相同"的回答，长度不同时其 log 概率如何变化
per_token = -0.5                                     # 假设每 token 平均 log 概率
for L in (50, 100, 200, 400):
    lp = per_token * L
    print(f"  长度 {L:>4}：log π = {lp:>8.1f}")
print("→ 长度直接进入 log 概率：不做长度归一化，DPO 会在'更短的回答'上系统性给低分。")
print("  SimPO 的 β/|y| 除以长度、DPO 的成对长度匹配，都是在拆这颗雷。")


## 6. 小结与下钻

- **DPO = 闭式解 + BT，配分函数被配对相减消掉** → 不需要 RM，也不需要 rollout。
- **代价是不会探索**：被静态数据上界框住；数学/代码这类"要超出数据"的任务提升有限。
- **β 决定隐式奖励的尺度**；`σ(−Δ)` 自动降权已分对样本（从 BT 继承）。
- **家族变体在拆三个约束**：要不要参考模型、要不要成对、要不要长度归一化。

下一站：[环节 07 · GRPO 与 RLVR](./环节07-GRPO与RLVR详解.md)（回到在线，但删掉 critic、把奖励换成验证器）。
